In [ ]:
from dotenv import load_dotenv
load_dotenv()


True

In [3]:

from openai import OpenAI
openai_client = OpenAI()

In [4]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.deepseek.com"
)

In [5]:
def llm(prompt):
    response = openai_client.responses.create(
        model="deepseek-flash",
        input=prompt
    )
    return response.output_text

In [6]:
llm("Hey, what's up?")

"Hey! Not much—just here and ready to help. What's up with you?"

In [ ]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

I don’t have access to your course’s enrollment info, so I can’t confirm for sure. In general:

- If it’s **self-paced/open enrollment**, you can usually join anytime.
- If it’s a **live cohort or fixed-schedule course**, you may still be able to join late, but it depends on the course policy.
- Check the course page for an **Enroll/Join** button or a **late enrollment** note.
- If enrollment looks closed, contact the instructor or course support and ask if late joining is possible.

If you tell me the course name or platform, I can help you figure out the next step.


In [8]:
context = '''
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

edit on GitHub
#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

edit on GitHub
#What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram and Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.

Don’t post questions in chat as they may be missed if the room is very active.

edit on GitHub
#How should I start the course and follow the weekly workflow?
Start with the LLM Zoomcamp docs, the general Zoomcamp logistics docs, and the LLM Zoomcamp GitHub repository.

You can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the course management platform.

A typical workflow is:

Watch the lesson videos.
Work through the lesson notebooks/code.
Read the homework instructions on GitHub.
Submit answers through the course platform before the deadline.
Homework is similar to the lesson flow, but uses a different dataset or slightly different task.

'''

In [10]:
prompt = f'''
Your task is to answer questions from the course participants based on the provided context. 
Use the context to find relevant information and provide accurate answers. If the answer is not
found in the context, respond with "I don't know". 

Question: 
{question}

Context: 
{context}
'''

In [12]:
print(prompt)


Your task is to answer questions from the course participants based on the provided context. 
Use the context to find relevant information and provide accurate answers. If the answer is not
found in the context, respond with "I don't know". 

Question: 
I just discovered the course. Can I join now?

Context: 

I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

edit on GitHub
#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

edit on GitHub
#What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
The zoom link is only published to instructo

In [13]:
question = "I just discovered the course. Can I join now?"
answer = llm(prompt)
print(answer)

Yes, you can join now. You can start whenever you want — the videos and GitHub materials are available. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [ ]:
# What we just did was naive. I knew in advance which FAQ entry held the answer and pasted it in by hand. What we want instead is to perform search automatically. We take the student's question, find the most relevant documents, and send those to the LLM.

# In code, it looks like this:

# def rag(question):
#     search_results = search(question)
#     user_prompt = build_prompt(question, search_results)
#     return llm(user_prompt) 

In [ ]:
import requests
#  FAQ data is available as JSON, data available at a JSON endpoint we can fetch directly
docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [15]:
courses_raw

[{'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 48},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 460},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 397},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 153},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 249}]

In [ ]:
documents = []
url_prefix = "https://datatalks.club/faq"
# Add questions from all courses in one list
for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}""" # Full path

    course_response = requests.get(course_url)
    course_response.raise_for_status() # If sth is broken, raise an error
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1400

In [20]:
documents[10]

{'id': '310a39c0f9',
 'course': 'ai-dev-tools-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Can I take the course on a locked-down work laptop without admin rights?',
 'answer': 'Yes. Use GitHub Codespaces - you only need a modern browser (or VS Code) and an internet connection, and you can install whatever you need inside the Codespace. GitHub Copilot and OpenAI Codex both work in Codespaces. The free tier is usually generous enough for the course.'}

In [ ]:
# Each course has a slug - a short identifier used in URLs. For example, machine-learning-zoomcamp, data-engineering-zoomcamp, etc.
#  We'll use these slugs for filtering in search.

In [ ]:
# We'll index the question, section, and answer fields as text (they'll be tokenized and ranked),
# and the course field as a keyword (for filtering).
# Text fields are the fields you search through. When you type a query, the search engine looks at these fields and tokenizes them. 
# It splits text into words, lowercases them, removes stop words, and ranks the results by relevance. 
# Keyword fields are for exact matching. Think of a SQL query like SELECT * FROM index WHERE course = 'data-engineering-zoomcamp'.
# The results must come from the specified course, no matter what ranking or boosting you do for text fields.
# You use keyword fields to restrict the search space to a particular subset. In our case, we have four courses.
# Say you're taking the LLM course and ask a question. You don't want answers from the MLOps or machine learning courses mixed in.



In [27]:
# Build an Index

from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)
index.fit(documents)

In [ ]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

  

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

In [ ]:
# Not all fields are equally important. The question field is usually more relevant than section for matching. 
# Query words appearing in the question is a stronger signal than them appearing in the section name.

[]

In [29]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [30]:
search_results = search(question)

In [ ]:
# Instruction prompt is usually fixed

INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [ ]:
# Change user prompt as per question

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [34]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

In [ ]:
# Build LLM context from the search results returned from indexing the document

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [37]:
context = build_context(search_results)

In [39]:
print(context)

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a total number of hours.

Module 3: Orchestration
Q: Why do we need orchestration / Kestra — can't I just run the code in a notebook?
A: Notebooks are great for learning and experimenting, but real AI workflows need more than a script that runs once: scheduling, retri

In [40]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [41]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a total number of hours.

Module 3: Orchestration
Q: Why do we need orchestration / Kestra — can't I just run the code in a notebook?
A: Notebooks are great for learning and experimenting, but real AI w

In [ ]:
# 2 OpenAI APIs - Chat Completions and Responses (Newest and Best)

response = openai_client.responses.create(
    model="deepseek-flash",
    input=prompt
)

In [ ]:
# The response is a Pydantic object. The answer is in response.output - a list of output items.
response.output_text

'Yes — you can still join now. You don’t need a confirmation email or registration to start; you can begin learning and submitting homework while the forms are open.\n\nIf you want the certificate: you need to complete a capstone project and the required peer reviews while a live cohort is still accepting submissions. Homework is not required for the certificate. So you can join and learn now, but certificate eligibility depends on the live cohort timeline.'

In [47]:
response.output[0].content[0].text

'We need answer user. Need use context. User asks: "I just discovered the course. Can I join now?" Context has similar Q/A: "I just discovered the course. Can I still join? A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions." Also registration not needed etc. Need answer likely: Yes, you can join now. You can start learning and submitting homework while form open; no confirmation email needed. If want certificate, must complete capstone project and peer reviews while live cohort accepting submissions. Self-paced alone doesn\'t get certificate. Maybe mention if live cohort currently accepting. Need maybe concise. Could answer with nuance: Yes, you can join/start now. If it\'s live cohort still accepting, you can submit project for certificate. If not, you can still learn self-paced but certificate requires live cohort. Need maybe mention no registration needed? The context: Q: registered for LLM Zoomcamp confirmation

In [48]:
response.usage

ResponseUsage(input_tokens=533, input_tokens_details=InputTokensDetails(cache_write_tokens=None, cached_tokens=0), output_tokens=400, output_tokens_details=OutputTokensDetails(reasoning_tokens=311), total_tokens=933)

In [49]:
# Price per 1M tokens https://api-docs.deepseek.com/quick_start/pricing

input_price = 0.003/1000000 
output_price = 0.6/1000000

cost = response.usage.input_tokens*input_price + response.usage.output_tokens*output_price


In [50]:
cost

0.00024159899999999998

In [51]:
model = "deepseek-flash"

In [52]:
# Think of a ChatGPT conversation. It starts with a hidden system prompt that tells the LLM how to behave, one you never see. After that, 
# your messages and the LLM's replies alternate. The LLM has no memory of its own, so it needs the full history passed in to continue the conversation.



message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model=model,
    input=message_history
)

In [53]:
def llm(instructions, user_prompt, model=model):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [54]:
def rag(query, model=model):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [55]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


Minsearch is in-memory. It's a bunch of Python dictionaries bound to the process where it's running. When you stop the process, the data disappears, so you re-index every time you restart. That's wasteful if indexing is slow or the data takes time to prepare.

So we separate ingestion from querying. One process writes the data to a persistent search index. Another process reads from it. The two run independently and only share the index between them.

The index survives restarts, so we ingest once and query as often as we like. This is the ingestion step from data engineering. We move data from its source into a target system the application can use.